# Portfolio Project 3: A Small Local RAG Assistant

> Build and evaluate a tiny retrieval workflow over a local knowledge base without external APIs.

**Track:** Machine Learning Roadmap · **Language:** English

## Project brief
Create a minimal retrieval-augmented question-answer helper for internal ML study notes. It retrieves text snippets with TF-IDF and returns evidence; it does not call an LLM. This gives you a testable retrieval foundation before adding a generator or vector database.

Do not use private documents in a public repository. A full RAG system also needs source-level access control, indexing strategy, evaluation data, model/prompt versioning, and a safe fallback when evidence is missing.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

knowledge_base = [
    {"id":"data-split", "text":"Split data into train, validation, and test partitions. Fit preprocessing on training data only."},
    {"id":"metrics", "text":"Precision measures the fraction of predicted positives that are true positives. Recall measures the fraction of actual positives found."},
    {"id":"pipeline", "text":"A preprocessing pipeline combines imputers, scalers, encoders, and an estimator to reduce leakage and support reproducibility."},
    {"id":"drift", "text":"Monitor input data quality, feature distributions, prediction behavior, latency, and model performance once labels arrive."},
]
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
X_docs = vectorizer.fit_transform([item["text"] for item in knowledge_base])

def retrieve(question, top_k=2):
    query = vectorizer.transform([question])
    scores = cosine_similarity(query, X_docs).ravel()
    order = scores.argsort()[::-1][:top_k]
    return [{"id":knowledge_base[i]["id"],"text":knowledge_base[i]["text"],"score":float(scores[i])} for i in order]

question="How can I prevent leakage from preprocessing?"
for hit in retrieve(question): print(hit)

{'id': 'pipeline', 'text': 'A preprocessing pipeline combines imputers, scalers, encoders, and an estimator to reduce leakage and support reproducibility.', 'score': 0.2804205161736727}
{'id': 'data-split', 'text': 'Split data into train, validation, and test partitions. Fit preprocessing on training data only.', 'score': 0.1116677586245759}


## Add a retrieval evaluation set
Evaluation needs questions, expected relevant source IDs, and a versioned corpus. Recall@k checks whether at least one expected source was retrieved. With a larger evaluation set, consider MRR or nDCG and review queries where retrieval fails.

In [2]:
evaluation_set = [
    {"question":"How should scaling be fitted?", "relevant_ids":{"data-split","pipeline"}},
    {"question":"What do precision and recall mean?", "relevant_ids":{"metrics"}},
    {"question":"What should be monitored in production?", "relevant_ids":{"drift"}},
]
rows=[]
for item in evaluation_set:
    hits=retrieve(item["question"],top_k=2)
    retrieved={h["id"] for h in hits}
    rows.append({"question":item["question"],"retrieved_ids":", ".join(sorted(retrieved)),"hit@2":bool(retrieved & item["relevant_ids"]),"recall@2":len(retrieved & item["relevant_ids"])/len(item["relevant_ids"])})
evaluation = pd.DataFrame(rows)
display(evaluation)
print("Mean hit@2:", evaluation["hit@2"].mean().round(3), "Mean recall@2:", evaluation["recall@2"].mean().round(3))

Mean hit@2: 1.0 Mean recall@2: 0.833


,question,retrieved_ids,hit@2,recall@2
0,How should scaling be fitted?,"drift, pipeline",True,0.5
1,What do precision and recall mean?,"drift, metrics",True,1.0
2,What should be monitored in production?,"drift, pipeline",True,1.0


In [3]:
def answer_with_evidence(question, min_score=0.05):
    hits = retrieve(question, top_k=2)
    useful = [h for h in hits if h["score"] >= min_score]
    if not useful:
        return {"answer":"I could not find enough evidence in the local notes.", "sources":[]}
    evidence = " ".join(hit["text"] for hit in useful)
    return {"answer":"Based only on the retrieved notes: " + evidence, "sources":[hit["id"] for hit in useful]}

print(answer_with_evidence("What is recall?"))
print(answer_with_evidence("How do I bake sourdough bread?"))

{'answer': 'Based only on the retrieved notes: Precision measures the fraction of predicted positives that are true positives. Recall measures the fraction of actual positives found.', 'sources': ['metrics']}
{'answer': 'I could not find enough evidence in the local notes.', 'sources': []}


## What would change for production RAG?
- Replace TF-IDF with a tested embedding model only if it improves retrieval on your evaluation set.
- Version the corpus, chunking policy, embedding model, index, prompt, and generator.
- Include citations and ensure claims are supported by retrieved content.
- Treat retrieved documents as untrusted input; they can contain malicious instructions.
- Evaluate retrieval quality, answer grounding, refusal behavior, latency, and cost separately.

## Portfolio deliverable
Include a diagram of ingestion → chunking → indexing → retrieval → evidence-aware response → evaluation; publish a small non-sensitive evaluation set and discuss failure cases.